Deflating spatially: is the north poor, or just cheap?
======================================================

**Author:** Ethan Ligon



## What this is



Session 2 computed one poverty line for the whole of Ghana and applied it
everywhere.  Food is not the same price everywhere, so a cedi buys more in
some regions than others, and the headcount in a cheap region overstates
its poverty.  Deaton and Zaidi's answer is a spatial price index built from
the survey's own unit values, and this notebook builds one.

The punchline: the north–south gap in the food-purchase headcount is 31
percentage points before deflation and 27 after.  About nine tenths of it
survives.  Deflation is the right thing to do and it is not where the
gradient comes from.  The more interesting finding is what the unit values
turn out to measure, which is in section 2.

-   **Prerequisites:** `lsms_library` on the release kernel, GhanaLSS microdata.
    Self-contained; it doesn't assume `session2.ipynb` has run.



## Setup



Session 2's aggregate, weights, FGT function and placeholder line.
`food_total` is food *purchases* per household; the aggregate is per
household, not per adult equivalent, so that the only thing that changes
between "nominal" and "real" below is the deflator.  Both refinements are
left to the exercises.



In [1]:
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ghana = ll.Country('GhanaLSS')
wave = '2016-17'

food_total = ghana.food_expenditures().groupby(['t', 'i']).sum().squeeze()
sample = ghana.sample()

C = food_total.xs(wave, level='t')                 # cedi per household, purchases
s = sample.xs(wave, level='t').reindex(C.index)
w, region = s.weight, s.strata                     # strata carries the region name

def fgt(c, w, z, alpha=0):
    """Weighted FGT_alpha.  Sums over the poor only: see session 2 for why."""
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w)
    c, w = c[ok], w[ok]
    poor = c < z
    return np.sum(w[poor] * ((z - c[poor]) / z) ** alpha) / np.sum(w)

def by_region(c, w, z):
    df = pd.DataFrame({'c': c, 'w': w, 'r': region})
    return df.groupby('r').apply(lambda d: fgt(d.c, d.w, z))

z = C.quantile(0.25)                               # session 2's placeholder line
north = ['Northern', 'Upper East', 'Upper West']
print(f"national P_0 = {fgt(C, w, z):.4f} at z = {z:.2f}")

`strata` is the sampling stratum, which in GLSS7 is the region: the ten
regions Ghana had before the 2019 split.  "Northern" is therefore the old
Northern Region with Savannah and North East still inside it, and *north*
below means Northern, Upper East and Upper West.



## 1.  The gradient, in nominal cedi



In [1]:
nominal = by_region(C, w, z).sort_values()
print(nominal.round(3).to_string())

isn = region.isin(north)
gap = fgt(C[isn], w[isn], z) - fgt(C[~isn], w[~isn], z)
print(f"\nnorth {fgt(C[isn], w[isn], z):.3f}  south {fgt(C[~isn], w[~isn], z):.3f}"
      f"  gap {gap:.3f}")

Greater Accra at 3%, Upper West at 59%, and the three northern regions
together at 42% against 12% for the other seven.  That is the gradient the
rest of this notebook tries to deflate away.



## 1.  What a unit value is, and what it isn't



Purchases carry an expenditure and a quantity but no price, so the price is
their ratio, and it is comparable only within an item *and* a unit.  The
national median in each item-unit cell is the numeraire; the regional
median relative to it is the regional price.



In [1]:
pu = ghana.food_acquired().xs(wave, level='t').xs('purchased', level='s')
uv = (pu.Expenditure / pu.Quantity).replace([np.inf, -np.inf], np.nan).dropna()
uv = uv.to_frame('uv').join(region.rename('r'), on='i')

p0 = uv.groupby(['j', 'u']).uv.median()            # national median, per item-unit
pr = uv.groupby(['r', 'j', 'u']).uv.median()       # regional median
rel = pr / p0.reindex(pr.index.droplevel('r')).values

top = uv.groupby(['j', 'u']).size().sort_values(ascending=False).head(8)
print(f"{len(uv)} unit values in {len(p0)} item-unit cells; "
      f"{len(pr)} region-item-unit cells")
print("\nregional price relative to national, eight most-reported cells:")
print(rel.unstack('r').loc[top.index].T.round(2).to_string())

Look at the units.  The most-reported cells are tomatoes, onions, pepper,
herring and okra by the *heap*, bread by the *loaf*, and kenkey and cooked
rice by *value*, where the quantity is the expenditure and the unit value
is one by construction.  A heap is whatever one cedi buys, so its unit
value is 1.0 or 2.0 nearly everywhere and the regional relatives are
mostly 1.0 or 2.0.  The survey did not record a heap getting smaller in
Accra; it recorded that a heap still costs one cedi.  Spatial price
variation for these goods lives in the size of the heap, which the survey
does not carry.

**Exercise 2.1.** Restrict to metric units (`Kg`, `Liter`, `Gram`,
`Milliliter`) and reprint the table for the eight most-reported metric
cells.  Do the regional relatives look like prices now?

**Exercise 2.2.** For `Bread` by the `Loaf`, Northern and Volta come in at
half the national median.  Is that a cheaper loaf or a smaller one?  You can't tell from this table; say what
you'd need, and check whether `community_prices()` carries it.



## 1.  A regional Paasche index



Paasche weights each region's own bundle: the ratio of what the region's
purchases cost at its own prices to what they'd cost at national prices.
It is the index that answers "how much more would this region's households
have paid for what they actually bought, elsewhere".



In [1]:
Q = (pu.Quantity.groupby(['i', 'j', 'u']).sum().to_frame('Q')
       .join(region.rename('r'), on='i')
       .groupby(['r', 'j', 'u']).Q.sum())          # the regional bundle

bundle = (Q.to_frame('Q').join(pr.rename('pr'))
            .join(p0.rename('p0'), on=['j', 'u']).dropna())
paasche = bundle.groupby('r').apply(lambda d: (d.pr * d.Q).sum() / (d.p0 * d.Q).sum())

E = pu.Expenditure.groupby(['i', 'j', 'u']).sum()
covered = E[E.index.droplevel('i').isin(p0.index)].sum() / E.sum()
print(f"share of purchase expenditure in cells with a unit value: {covered:.3f}")
print(paasche.sort_values().round(3).to_string())

Upper West at 0.86, Greater Accra at 1.11, and the three northern regions
all below one.  Every cedi of purchase expenditure sits in a cell with a
national median, so nothing is dropped; but section 2 says the heap-priced
goods contribute almost nothing to the dispersion, so this is a lower bound
on the spatial variation in food prices, not an estimate of it.

**Exercise 3.1.** Build the Laspeyres counterpart, the *national* bundle at
regional prices.  It comes out much larger than the Paasche in every region
and the reason is in the sparse cells: a region that rarely buys an item
still gets a regional median for it, from a handful of households.  Show
this, then decide how many observations a region-item-unit cell needs
before you believe its median, and rebuild both indices.



## 1.  Deflate, and see what survives



Divide each household's nominal purchases by its region's index.  The
index is normalised to national median prices, and the line is a quantile
of nominal consumption, so the line stays where it was.



In [1]:
Creal = C / paasche.reindex(region).values
real = by_region(Creal, w, z)

out = pd.DataFrame({'index': paasche, 'nominal': nominal, 'real': real}).sort_values('nominal')
print(out.round(3).to_string())

gap_real = fgt(Creal[isn], w[isn], z) - fgt(Creal[~isn], w[~isn], z)
print(f"\nnational P_0: nominal {fgt(C, w, z):.4f}  real {fgt(Creal, w, z):.4f}")
print(f"north-south gap: nominal {gap:.3f}  real {gap_real:.3f}"
      f"  ({100 * gap_real / gap:.0f}% survives)")

Upper West drops seven points and Northern three; Greater Accra rises one.
The gap between north and south goes from 30.5 to 27.3 points, so 89% of
the nominal gradient survives a Paasche deflation built from the survey's
own unit values.  If you were hoping the north was merely cheap, the data
don't oblige.



In [1]:
fig, ax = plt.subplots(figsize=(6.5, 4))
y = np.arange(len(out))
ax.hlines(y, out.nominal, out.real, color='#b0b0b0', lw=2)
ax.scatter(out.nominal, y, color='#b0b0b0', s=40, zorder=3, label='nominal')
ax.scatter(out.real, y, color='#1f4e79', s=40, zorder=3, label='deflated (Paasche)')
ax.set_yticks(y); ax.set_yticklabels(out.index)
ax.set_xlabel('weighted headcount ratio')
ax.set_title('Regional headcount before and after spatial deflation')
ax.legend(frameon=False, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

**Exercise 4.1.** The line was held fixed.  The alternative is to re-take
the lower quartile of *deflated* consumption.  Do that and report the
gap.  It should be nearly the same, and you should be able to say why
before running it.

**Exercise 4.2.** The index is regional.  Build it at the cluster level `v`
instead, falling back to the region when a cluster-item-unit cell is
empty, and report what happens to the within-region spread of the
headcount.



## Exercises



1.  Everything here is on purchases, and the `food_sources` notebook shows
    own production is 29% of food value nationally and more in the north.
    Add it, valued at the national median unit value, and redo section 4.
    Which does more to the gradient, valuing own production or deflating?
2.  Section 2 found that heap-priced goods carry no spatial price
    information.  Rebuild the Paasche index from metric-unit cells only and
    say whether the deflated gradient changes.  Then say which of the two
    indices you'd publish, and why a reader should believe the one you
    picked.
3.  Per household versus per adult equivalent.  Northern households are
    larger.  Redo sections 1 and 4 on $C_i / A_i$ with the scale from the
    `equivalence_scales` notebook.  Between household size and prices, which
    explains more of the north–south gap?

